In [1]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

from pymilvus.model.hybrid import BGEM3EmbeddingFunction
from sentence_transformers import SentenceTransformer


In [2]:
import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Add the EDP directory to the Python path
# sys.path.append(os.path.abspath(os.path.join('..', 'EDP')))

In [3]:
from parser_2 import Parser2

In [5]:
xml_file_path = '/home/sbasir/Thesis/Thesis/EDP/misc/output_solr34.xml'

raw_data = Parser2.XLMtoString(xml_file_path)

In [6]:
display(raw_data[:10])

only_text = [x['text'] for x in raw_data]

display(only_text[:10])

[{'id': 'ID: /2021672/resource_document_mauritshuis_397',
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340',
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_426',
  'text': 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_432',
  'text': 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_354',
  'text': 'St Barbara by Parmigianino, dated None'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_181',
  'text': "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737"},
 {'id': 'ID: /2021672/resource_document_mauritshuis_195',
  'text': 'A Fishmo

['Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660',
 'The Annunciation by Francesco Solimena, dated 1693 - 1693',
 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611',
 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579',
 'St Barbara by Parmigianino, dated None',
 "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737",
 'A Fishmonger at the Door by Jacob Ochtervelt, dated 1663 - 1663',
 'Portrait of a Deceased Girl, probably Catharina Margaretha van Valkenburg by Johannes Thopas, dated 1682 - 1682',
 'View of the Dam and the Damrak in Amsterdam by Jacob van Ruisdael, dated 1675 - 1672',
 'Hall Settle by Anonymous (Northern Netherlands), dated 1720 - 1700']

In [36]:
limit = 128

def chunker(contexts: list):
    chunks = []
    all_contexts = ' '.join(contexts).split('.')
    chunk = []
    for context in all_contexts:
        chunk.append(context)
        if len(chunk) >= 3 and len('.'.join(chunk)) > limit:
            # surpassed limit so add to chunks and reset
            chunks.append('.'.join(chunk).strip()+'.')
            # add some overlap between passages
            chunk = chunk[-2:]
    # if we finish and still have a chunk, add it
    if chunk is not None:
        chunks.append('.'.join(chunk))
    return chunks

chunks = chunker(only_text)
chunks

ids = []
for i in range(len(chunks)):
    ids.append(i)

chunked_data = []
for raw in raw_data:
    chunks = chunker([raw['text']])
    for i, chunk in enumerate(chunks):
        chunk_id = f'{raw["id"]}_{i}'
        chunked_data.append({'id': chunk_id, 'text': chunk})

chunked_data[:10]


[{'id': 'ID: /2021672/resource_document_mauritshuis_397_0',
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340_0',
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_426_0',
  'text': 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_432_0',
  'text': 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_354_0',
  'text': 'St Barbara by Parmigianino, dated None'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_181_0',
  'text': "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737"},
 {'id': 'ID: /2021672/resource_document_mauritshuis_195_0',
  'te

In [7]:
# Generate embeddings using BGEM3 model
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
embeddings = model.encode([x['text'] for x in raw_data])

/home/sbasir/Thesis/myenv/lib/python3.12/site-packages/torch/cuda/__init__.py:118: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.69k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

2_Dense/config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

In [8]:
display(embeddings[0])

type(embeddings[0])

array([ 0.00862602,  0.0128614 , -0.03376903, -0.0624555 ,  0.03864206,
        0.05342465, -0.03670567,  0.04166102, -0.08377833, -0.01893639,
        0.04251879,  0.01255133, -0.03283902,  0.05781703,  0.00289622,
        0.00688297,  0.04334608,  0.00053363, -0.01224209,  0.05585311,
       -0.04602017, -0.00439992,  0.0212906 ,  0.0301004 , -0.01374634,
       -0.04415502, -0.00353214,  0.03320285, -0.04548392, -0.01821415,
       -0.02135391,  0.02513985,  0.01648743, -0.02361494,  0.03936537,
        0.01675223, -0.01877876,  0.00692668, -0.02258342,  0.0063069 ,
       -0.07917231,  0.01772378, -0.04769323, -0.06543087,  0.00158489,
        0.03766427, -0.06071008, -0.01223131, -0.06645195,  0.00556124,
       -0.01105868,  0.00908042,  0.02290435,  0.03782691,  0.0298376 ,
        0.03477437, -0.02688894,  0.01732579, -0.04402055, -0.04607914,
       -0.02115686, -0.05822203, -0.06271309,  0.01808993,  0.00977028,
       -0.05360792, -0.03887076, -0.01604172, -0.04952199,  0.03

numpy.ndarray

In [10]:
entities = [
    [item['id'] for item in raw_data], #IDs
    [item['text'] for item in raw_data], #Text
    embeddings #Embeddings
]

In [11]:
from pymilvus import MilvusClient
from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

# client = MilvusClient("milvus_demo.db")

In [12]:
connections.connect("default", host="localhost", port="19530")

In [13]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=100),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=512),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=512),  # Ensure the dimension matches your embeddings
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'sbert_demo'
col = Collection(col_name, schema, consistency_level="Strong")

In [14]:
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()

In [305]:
# if col.has_collection(col_name):
#     print("Collection exists.")
    
# # # Create an index for the dense vector field
# # dense_index = client.prepare_index_params(index_type="IVF_FLAT", metric_type="L2", params={"nlist": 1024})

# # client.create_index(col_name, dense_index)

In [ ]:
# sparse_index = client.prepare_index_params(index_type="FLAT", metric_type="JACCARD", params={"nlist": 1024})
# client.create_index("sparse_vector", sparse_index)

In [15]:
col.insert(entities)

(insert count: 831, delete count: 0, upsert count: 0, timestamp: 452592003617652739, success count: 831, err count: 0

In [16]:
col.flush()

In [17]:
query = "Vermeer"
query_embeddings = model.encode(query)
k=10

# display(query_embeddings)

# print(query_embeddings)

search_params = {"metric_type": "IP"}

# Search topK docs based on dense and sparse vectors and rerank with RRF.
res = col.search(
    data = [query_embeddings],
    anns_field="dense_vector",
    param=search_params,
    limit=10,
    output_fields=["pk","text"]
)

for result in res[0]:
    print(result)

id: ID: /2021672/resource_document_mauritshuis_92, distance: 0.23221732676029205, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_92', 'text': 'View of Delft by Johannes Vermeer, dated 1661 - 1660'}
id: ID: /2021672/resource_document_mauritshuis_406, distance: 0.2102731168270111, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_406', 'text': 'Diana and her Nymphs by Johannes Vermeer, dated 1654 - 1653'}
id: ID: /2021672/resource_document_mauritshuis_59, distance: 0.20405489206314087, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_59', 'text': "The Raven Robbed of the Feathers He Wore to Adorn Himself by Melchior d' Hondecoeter, dated 1671 - 1671"}
id: ID: /2021672/resource_document_mauritshuis_670, distance: 0.19617560505867004, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_670', 'text': 'Girl with a Pearl Earring by Johannes Vermeer, dated 1665 - 1665'}
id: ID: /2021672/resource_document_mauritshuis_299, distance: 0.1868467628955841, entit